## 04. scratchpad

### 1. Data Upload

In [42]:
import pandas as pd
import os
from collections import defaultdict 
import re

In [43]:
def upload_to_dataframe(root, files, num_lines):
    path_eng, path_pol = [root+f for f in files]
    data = defaultdict(list)
    
    with open (path_eng, encoding="utf_8") as f_eng, open (path_pol, encoding="utf_8") as f_pol:
        for _ in range(num_lines):
            data['eng_text'].append(f_eng.readline().strip())
            data['pol_text'].append(f_pol.readline().strip())
    return pd.DataFrame(data)

In [44]:
root = "../data/opus_opensub/en-pl.txt/"
files = ('OpenSubtitles.en-pl.en', '/OpenSubtitles.en-pl.pl')
df = upload_to_dataframe(root, files, 600000)

### 2. EDA & Sanitazation

#### 2.1 English Non-Ascii Sentences

In [45]:
def nonascii_list(df_series, is_pol):
    if is_pol:
        pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
        requir = lambda x: x.isascii() or x in pl_chars
    else:
        requir = lambda x: x.isascii()
        
    text_all = ''.join(df_series)
    unique_dct = {x for x in text_all if not requir(x)}
    return sorted(list(unique_dct))

In [46]:
print(nonascii_list(df['eng_text'], False))

['\x80', '\x81', '\x82', '\x83', '\x84', '\x85', '\x88', '\x8b', '\x8c', '\x8e', '\x8f', '\x91', '\x94', '\x96', '\x98', '\x99', '\x9c', '\x9d', '\xa0', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '¬', '\xad', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '¼', '½', '¾', '¿', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Ç', 'É', 'Ê', 'Ì', 'Ï', 'Ð', 'Ñ', 'Ô', 'Ö', 'Ø', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'ç', 'è', 'é', 'ê', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'Ă', 'ď', 'ę', 'Ł', 'ń', 'ő', 'œ', 'ś', 'Š', 'š', 'Ј', 'С', 'й', '،', 'أ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ح', 'ر', 'س', 'ش', 'ع', 'غ', 'ف', 'ك', 'ل', 'م', 'ن', 'و', 'ي', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', '\u200b', '‒', '–', '—', '‚', '€', '™', '─', '☻', '♥', '♪', '慹', '拢', '檛', '鈥', 'ﬁ', 'ﬂ']


In [47]:
is_ascii = lambda text: text.isascii()
maska = df['eng_text'].apply(is_ascii)
df_non_ascii = df[~maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.head()

(4595, 2)


,eng_text,pol_text
0,¿You had never seen A crab suedes? Them there ...,- Nigdy wcześniej nie widziałaś raków?
1,"He hears ¿, where you live?",Nigdy wcześniej nie byłam w lesie.
2,"Boys ¿, they are there?",Jesteście tam chłopaki?
3,- Te atrapé. - No es gracioso. Your he catches...,- Ale twoje mina była.
4,"Your also ¿, it is not like that?","- Tak, niezła jest"


In [48]:
df_2 = df[maska].reset_index(drop=True)
df_2.shape

(595405, 2)

#### 2.2 Polish Non-Ascii Sentences

In [50]:
pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
requir = lambda let: not let.isascii() and let not in pl_chars
maska = df_2['pol_text'].apply(lambda snt: any([requir(let) for let in snt]))
df_non_ascii = df_2[maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.sample(10)

(1227, 2)


,eng_text,pol_text
766,Buy up every minute of every radio station in ...,Wykup každa minute w každej podleglej ci stacj...
1223,"On the blackboard is drawn, Mihailo Potapych, ...","Jest tam narysowane, Mihhailo Potapõtð, głuchy..."
224,"- You are, aren't you?",– Istotnie. Tak przy okazji:
68,The Church is opening her arms to you if you t...,Ko¶ciół wyci±ga do ciebie rękę je¶li j± odrzuc...
837,"The world is not as bad as you think, Tom.","Tom, świat nie jest taki zły jak ​​myślisz."
287,- It's nothing at all.,– Niezupełnie.
944,My Fuhrer!,Meine führer!
658,I was hoping you'd be spared all this.,"Myslalem, že ci tego oszczedze."
215,- There's no need to hide it.,"– Nao-san, mów szczerze."
735,"I accuse this man. By his tone, by his careful...","Oskaržam tego czlowieka, že umiejetnie dobiera..."


In [55]:
df_3 = df_2[~maska].reset_index(drop=True)
print(df_3.shape)
df_3.sample(5)

(594178, 2)


,eng_text,pol_text
470445,"- Oh, a lot of things. He made some crack abou...","- Wiele rzeczy żartował sobie, że jestem mamin..."
216227,- Just want a table for two.,- Po prostu chce stolik dla dwoch osob.
298936,Nearer.,Bliżej.
148899,"Just this way, please.","Tędy, proszę."
551495,"Oh, you remembered.","Och, pamiętasz."


In [60]:
print(nonascii_list(df_3['eng_text'], False))
print(nonascii_list(df_3['pol_text'], True))

[]
[]


In [67]:
df_3.iloc[74182]['eng_text']

'The next time I...'

In [68]:
df_3.iloc[74182]['pol_text']

'Następnym razem...'

#### 2.3 Dialog like sentences [starts with - ]

In [ ]:
is_dialog

In [84]:
test = "  -Mason, Indiański agent."
bool(re.match(r"^ *- *", test))

True

In [69]:
df_3.sample(30)

,eng_text,pol_text
249374,"Here's your coffee, Danny.","Twoja kawa, Danny."
227566,And what do we do now?,Co teraz robimy?
443858,- I done my part of that job.,- Zrobiłem swoją część pracy.
554631,- So he can divorce you?,- A on może to zrobić?
452452,"Sloan, I take everything back.",Cofam wszystko.
23947,I think you'll find we have everything.,"Sądzę, że chyba mamy tu wszystko."
28852,And- - A poncho?,- Poncho?
330514,"Well, I'm as much against killing as ever, sir.",Nadal jestem przeciwny zabijaniu.
43644,- I can't either.,- Ja sama nie mogę.
267130,Now I can control life absolutely.,Teraz mogę całkowicie kontrolować życie.
